## Silver Transform
Converts both Bronze sources into the same canonical `header` and `line_items` shape, preserves lineage, then unions CSV and JSON together for downstream semantic validation.

In [0]:
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import col, trim, regexp_replace, when, row_number, from_json, explode_outer, split, size, lit, to_date, coalesce, regexp_extract, upper

In [0]:
# VALID_CSV_BRONZE_PATH = dbutils.jobs.taskValues.get(
#     taskKey="bronze_ge",
#     key="valid_csv_path",
#     debugValue=""
# )
# VALID_JSON_BRONZE_PATH = dbutils.jobs.taskValues.get(
#     taskKey="bronze_ge",
#     key="valid_json_path",
#     debugValue=""
# )

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

VALID_CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/validated/csv_raw/"
VALID_JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/validated/json_raw/"

VALID_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/transform/standardized/header/"
VALID_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/transform/standardized/lines/"
SILVER_MERGE_AUDIT_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/transform/merge_policy_audit/"

In [0]:
def _clean_numeric_str(col_name):
    return regexp_replace(trim(col(col_name).cast("string")), r"[$,%\s,]", "")


def _safe_decimal(col_name):
    cleaned = _clean_numeric_str(col_name)
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("decimal(18,2)"))


def _safe_percent(col_name):
    cleaned = _clean_numeric_str(col_name)
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(
        when(cleaned.cast("double") > 1, (cleaned.cast("double") / 100.0)).otherwise(cleaned.cast("double")).cast("decimal(9,6)")
    )


def _safe_int(col_name):
    cleaned = regexp_replace(trim(col(col_name).cast("string")), r"[,\s]", "")
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("double").cast("int"))


def _safe_date(col_name):
    raw = trim(col(col_name).cast("string"))
    return coalesce(
        F.try_to_date(raw, "yyyy-MM-dd"),
        F.try_to_date(raw, "M/d/yyyy"),
        F.try_to_date(raw, "MM/dd/yyyy"),
        F.try_to_date(raw, "d-MMM-yyyy"),
    )


def _trim_and_nullify(col_name):
    raw = trim(col(col_name).cast("string"))
    return when(raw == "", None).otherwise(raw)

def _upper_case_trim(col_name):
    return upper(trim(regexp_replace(col(col_name).cast("string"), r"\s+", " ")))

In [0]:
valid_csv_bronze_df = spark.read.format("delta").load(VALID_CSV_BRONZE_PATH)
valid_json_bronze_df = spark.read.format("delta").load(VALID_JSON_BRONZE_PATH)

if valid_csv_bronze_df.rdd.isEmpty():
    raise RuntimeError("Silver transform cannot start because validated Bronze CSV is empty.")
if valid_json_bronze_df.rdd.isEmpty():
    raise RuntimeError("Silver transform cannot start because validated Bronze JSON is empty.")

analyze_schema = spark.read.json(
    valid_json_bronze_df.select("analyzeResult").where(col("analyzeResult").isNotNull()).limit(50).rdd.map(lambda row: row[0])
).schema

if "analyzeResult" in valid_json_bronze_df.columns and dict(valid_json_bronze_df.dtypes)["analyzeResult"] == "string":
    valid_json_bronze_df = valid_json_bronze_df.withColumn("analyzeResult", from_json(col("analyzeResult"), analyze_schema))

In [0]:
csv_header_df = (
    valid_csv_bronze_df
    .withColumn("InvoiceId", _trim_and_nullify("InvoiceId"))
    .withColumn("OrderDate", _safe_date("OrderDate"))
    .withColumn("CustomerName", _trim_and_nullify("CustomerName"))
    .withColumn("ShipPostalCode", _trim_and_nullify("ShipPostalCode"))
    .withColumn("ShipCity", _trim_and_nullify("ShipCity"))
    .withColumn("ShipState", _trim_and_nullify("ShipState"))
    .withColumn("ShipCountry", _trim_and_nullify("ShipCountry"))
    .withColumn("ShipMode", _upper_case_trim("ShipMode"))
    .withColumn("BalanceDue", _safe_decimal("BalanceDue"))
    .withColumn("SubTotal", _safe_decimal("SubTotal"))
    .withColumn("DiscountPercent", _safe_percent("DiscountPercent"))
    .withColumn("DiscountAmount", _safe_decimal("DiscountAmount"))
    .withColumn("ShippingAmount", _safe_decimal("ShippingAmount"))
    .withColumn("InvoiceTotal", _safe_decimal("InvoiceTotal"))
    .withColumn("OrderId", _trim_and_nullify("OrderId"))
    .groupBy("InvoiceId")
    .agg(
        F.first("_source_file", ignorenulls=True).alias("_source_file"),
        F.first("OrderDate", ignorenulls=True).alias("OrderDate"),
        F.first("CustomerName", ignorenulls=True).alias("CustomerName"),
        F.first("ShipPostalCode", ignorenulls=True).alias("ShipPostalCode"),
        F.first("ShipCity", ignorenulls=True).alias("ShipCity"),
        F.first("ShipState", ignorenulls=True).alias("ShipState"),
        F.first("ShipCountry", ignorenulls=True).alias("ShipCountry"),
        F.first("ShipMode", ignorenulls=True).alias("ShipMode"),
        F.first("BalanceDue", ignorenulls=True).alias("BalanceDue"),
        F.first("SubTotal", ignorenulls=True).alias("SubTotal"),
        F.first("DiscountPercent", ignorenulls=True).alias("DiscountPercent"),
        F.first("DiscountAmount", ignorenulls=True).alias("DiscountAmount"),
        F.first("ShippingAmount", ignorenulls=True).alias("ShippingAmount"),
        F.first("InvoiceTotal", ignorenulls=True).alias("InvoiceTotal"),
        F.first("OrderId", ignorenulls=True).alias("OrderId"),
    )
    .withColumn("source_type", lit("csv"))
    .select(
        "_source_file", "source_type", "InvoiceId", "OrderDate", "CustomerName",
        "ShipPostalCode", "ShipCity", "ShipState", "ShipCountry", "ShipMode",
        "BalanceDue", "SubTotal", "DiscountPercent", "DiscountAmount",
        "ShippingAmount", "InvoiceTotal", "OrderId",
    )
)

csv_line_window = Window.partitionBy("InvoiceId").orderBy(
    col("ProductName").asc_nulls_last(),
    col("ProductId").asc_nulls_last(),
    col("_source_file").asc_nulls_last(),
)
csv_lines_df = (
    valid_csv_bronze_df
    .withColumn("InvoiceId", _trim_and_nullify("InvoiceId"))
    .withColumn("ProductName", _trim_and_nullify("ProductName"))
    .withColumn("SubCategory", _trim_and_nullify("SubCategory"))
    .withColumn("Category", _trim_and_nullify("Category"))
    .withColumn("ProductId", _trim_and_nullify("ProductId"))
    .withColumn("Quantity", _safe_int("Quantity"))
    .withColumn("UnitPrice", _safe_decimal("UnitPrice"))
    .withColumn("ItemSubTotal", _safe_decimal("ItemSubTotal"))
    .withColumn("source_type", lit("csv"))
    .withColumn("LineNumber", row_number().over(csv_line_window))
    .select(
        "_source_file", "source_type", "InvoiceId", "LineNumber", "ProductName",
        "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice", "ItemSubTotal",
    )
)

In [0]:
doc0 = col("analyzeResult.documents")[0]
fields = doc0["fields"]
content = col("analyzeResult").getField("content")
items_array = fields.getField("Items").getField("valueArray")

# Extract header fields using regex
ship_mode = upper(trim(regexp_replace(regexp_extract(content, r"Ship\s*Mode\s*:\s*\n?\s*([A-Za-z ]+)", 1), r"\s+", " ")))
raw_discount = regexp_extract(content, r"Discount\s*\(\s*(\d{1,3})\s*%\s*\)", 1)
discount_percent = when(raw_discount != "", raw_discount.cast("double") / 100.0)
raw_shipping = regexp_extract(content, r"Shipping\s*:\s*\$?\s*([0-9,]+\.\d{2})", 1)
shipping_amount = when(raw_shipping != "", regexp_replace(raw_shipping, ",", "").cast("decimal(18,2)"))
order_id = regexp_extract(content, r"Order\s*ID\s*:\s*([^\n\r]+)", 1)

# # Read one file to infer the full schema for analyzeResult
# analyze_schema = (
#     spark.read
#     .format("json")
#     .option("multiLine", "true")
#     .load(VALID_JSON_BRONZE_PATH)
#     .select("analyzeResult")
#     .schema["analyzeResult"].dataType
# )

# # Convert analyzeResult from string to struct if needed
# if valid_json_bronze_df.schema["analyzeResult"].dataType.typeName() == "string":
#     valid_json_bronze_df = valid_json_bronze_df.withColumn("analyzeResult", from_json(col("analyzeResult"), analyze_schema))

json_header_df = (
    valid_json_bronze_df
    .select(
        col("_source_file"),
        lit("json").alias("source_type"),
        fields.getField("InvoiceId").getField("valueString").alias("InvoiceId"),
        F.try_to_date(fields.getField("InvoiceDate").getField("valueDate")).alias("OrderDate"),
        fields.getField("CustomerName").getField("valueString").alias("CustomerName"),
        fields.getField("ShippingAddress").getField("valueAddress").getField("postalCode").alias("ShipPostalCode"),
        fields.getField("ShippingAddress").getField("valueAddress").getField("city").alias("ShipCity"),
        fields.getField("ShippingAddress").getField("valueAddress").getField("state").alias("ShipState"),
        fields.getField("ShippingAddress").getField("valueAddress").getField("countryRegion").alias("ShipCountry"),
        ship_mode.alias("ShipMode"),
        fields.getField("AmountDue").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("BalanceDue"),
        fields.getField("SubTotal").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("SubTotal"),
        discount_percent.cast("decimal(9,6)").alias("DiscountPercent"),
        fields.getField("TotalDiscount").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("DiscountAmount"),
        shipping_amount.alias("ShippingAmount"),
        fields.getField("InvoiceTotal").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("InvoiceTotal"),
        order_id.alias("OrderId"),
    )
)

json_lines_pre_df = (
    valid_json_bronze_df
    .select(
        col("_source_file"),
        lit("json").alias("source_type"),
        fields.getField("InvoiceId").getField("valueString").alias("InvoiceId"),
        explode_outer(items_array).alias("item"),
    )
    .select(
        "_source_file",
        "source_type",
        "InvoiceId",
        col("item").getField("valueObject").getField("Description").getField("valueString").alias("Description"),
        col("item").getField("valueObject").getField("Quantity").getField("valueNumber").cast("int").alias("Quantity"),
        col("item").getField("valueObject").getField("UnitPrice").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("UnitPrice"),
        col("item").getField("valueObject").getField("Amount").getField("valueCurrency").getField("amount").cast("decimal(18,2)").alias("ItemSubTotal"),
    )
    .withColumn("desc_lines", split(col("Description"), r"\r?\n"))
    .withColumn("ProductName", when(size(col("desc_lines")) >= 1, trim(col("desc_lines")[0])))
    .withColumn("meta_line", when(size(col("desc_lines")) >= 2, trim(col("desc_lines")[1])))
    .withColumn("meta_parts", when(col("meta_line").isNotNull(), split(col("meta_line"), r"\s*,\s*")))
    .withColumn("SubCategory", when(size(col("meta_parts")) >= 1, trim(col("meta_parts")[0])))
    .withColumn("Category", when(size(col("meta_parts")) >= 2, trim(col("meta_parts")[1])))
    .withColumn("ProductId", when(size(col("meta_parts")) >= 3, trim(col("meta_parts")[2])))
    .drop("desc_lines", "meta_line", "meta_parts")
)

json_line_window = Window.partitionBy("InvoiceId").orderBy(col("ProductName").asc_nulls_last(), col("ProductId").asc_nulls_last())
json_lines_df = (
    json_lines_pre_df
    .withColumn("LineNumber", row_number().over(json_line_window))
    .select(
        "_source_file", "source_type", "InvoiceId", "LineNumber", "ProductName",
        "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice", "ItemSubTotal",
    )
)


In [0]:
silver_header_df = csv_header_df.unionByName(json_header_df, allowMissingColumns=True)
silver_lines_df = csv_lines_df.unionByName(json_lines_df, allowMissingColumns=True)

# Identify InvoiceIds that are present in both sources to enforce merge policy and create audit records for collisions
csv_invoice_ids = silver_header_df.filter(col("source_type") == "csv").select("InvoiceId").distinct()
json_invoice_ids = silver_header_df.filter(col("source_type") == "json").select("InvoiceId").distinct()
colliding_ids_df = csv_invoice_ids.intersect(json_invoice_ids)

silver_merge_audit_df = (
    colliding_ids_df
    .withColumn("rule_id", lit("silver_merge_csv_priority"))
    .withColumn("severity", lit("WARNING"))
    .withColumn("winner_source_type", lit("csv"))
    .withColumn("loser_source_type", lit("json"))
    .withColumn("dq_reason", lit("InvoiceId collision across sources resolved with CSV priority"))
    .withColumn("issue_ts", F.current_timestamp())
)

silver_header_df = silver_header_df.join(
    colliding_ids_df.withColumnRenamed("InvoiceId", "_cid"),
    on=(col("InvoiceId") == col("_cid")) & (col("source_type") == lit("json")),
    how="left_anti",
).drop("_cid")
silver_lines_df = silver_lines_df.join(
    colliding_ids_df.withColumnRenamed("InvoiceId", "_cid"),
    on=(col("InvoiceId") == col("_cid")) & (col("source_type") == lit("json")),
    how="left_anti",
).drop("_cid")

if silver_header_df.rdd.isEmpty() or silver_lines_df.rdd.isEmpty():
    raise RuntimeError("Silver transform produced empty standardized datasets.")

silver_header_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(VALID_SILVER_HEADER_PATH)
silver_lines_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(VALID_SILVER_LINES_PATH)
silver_merge_audit_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MERGE_AUDIT_PATH)

dbutils.jobs.taskValues.set(key="valid_silver_header_path", value=VALID_SILVER_HEADER_PATH)
dbutils.jobs.taskValues.set(key="valid_silver_lines_path", value=VALID_SILVER_LINES_PATH)
dbutils.jobs.taskValues.set(key="silver_merge_audit_path", value=SILVER_MERGE_AUDIT_PATH)

In [0]:
# Databricks table registration for Metaplane: Silver transform outputs
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_SILVER_STANDARDIZED_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_standardized_header"
BATCH_SILVER_STANDARDIZED_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_standardized_lines"
BATCH_SILVER_MERGE_AUDIT_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_merge_policy_audit"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_STANDARDIZED_HEADER_TABLE}
USING DELTA
LOCATION "{VALID_SILVER_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_STANDARDIZED_LINES_TABLE}
USING DELTA
LOCATION "{VALID_SILVER_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_MERGE_AUDIT_TABLE}
USING DELTA
LOCATION "{SILVER_MERGE_AUDIT_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_SILVER_STANDARDIZED_HEADER_TABLE}"))
display(spark.sql(f"DESCRIBE DETAIL {BATCH_SILVER_STANDARDIZED_LINES_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,fdd8475e-ab82-4673-b26f-434d0afa434d,hant-catalog.invoice.batch_silver_standardized_header,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/silver/transform/standardized/header,2026-04-23T18:43:35.31Z,2026-04-23T18:43:39Z,List(),List(),5,147413,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,26c876ce-8022-4f56-b2ac-5308e6bac973,hant-catalog.invoice.batch_silver_standardized_lines,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/silver/transform/standardized/lines,2026-04-23T18:43:39.832Z,2026-04-23T18:43:43Z,List(),List(),2,82403,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
